# video-base-imagem-padrao.ipynb — Narrated Video (image background, standard)

Generates the **video base** using Pixabay PHOTOS (still images) as
background, **standard mode**: narration + credited/logoed photos (random
unused rows, DURACAO_CLIPE seconds each, default 5s) + background music.
No verse matching — for that, use `video-base-imagem-versiculo.ipynb`. No
subtitles yet either — that's what the next notebooks are for.

For a VIDEO-clip background instead, use `video-base-video-padrao.ipynb`.

**How to use:**
1. Run cells top to bottom.
2. Edit only the "⚙️ Configuration" cell to start a new video.
3. Each step skips automatically what's already done (checkpoint).


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System packages ──────────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# ── Python packages ───────────────────────────────────────────────────────────
!pip install -q edge-tts pandas gdown yt-dlp nest_asyncio gspread
print('✅ Python packages')

# ── Mount Drive (unmount first to avoid a stuck session) ────────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# ── Copy modules from Drive to /content/pipeline ────────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixed for the whole project (same value as Configuration)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")
    print("   Make sure the .py files are in pipeline/modulos/ on Drive.")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-18s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


✅ ffmpeg + espeak-ng
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 74.6 MB/s eta 0:00:00
✅ Python packages
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ Drive mounted
✅ 13 modules copied from /content/drive/MyDrive/narrated_video/pipeline/modulos
✅ Setup complete!


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell when starting a new video               ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY ──────────────────────────────────────────────────────
NOME_ORACAO = "40_Matt_02"           # short identifier, no spaces/accents
                                    # e.g. "40_matt_02", "genesis_01"

# ── 2. FULL TEXT (for Edge TTS) — ignored if a script already exists on Drive
TEXTO_ORACAO = (
"Now when Jesus was born in Bethlehem of Judea, in the days of King Herod. "
"Behold, wise men from the east came to Jerusalem. "
"Saying, Where is He who is born King of the Jews? "
"For we saw His star in the east and have come to worship Him. "
"When King Herod heard it, he was troubled. "
"And all Jerusalem with him. "
"Gathering together all the chief priests and scribes of the people. "
"He asked them where the Christ would be born. "
"They said to him: In Bethlehem of Judea, for thus it is written through the prophet. "
"You, Bethlehem, land of Judah, are in no way least among the princes of Judah. "
"For out of you shall come forth a governor who shall shepherd My people Israel. "
"Then Herod secretly called the wise men. "
"And learned from them exactly what time the star appeared. "
"He sent them to Bethlehem and said: Go and search diligently for the young child. "
"When you have found him, bring me word, so that I also may come and worship him. "
"They, having heard the King, went their way. "
"And behold, the star which they saw in the east went before them. "
"Until it came and stood over where the young child was. "
"When they saw the star, they rejoiced with exceedingly great joy. "
"They came into the house and saw the young child with Mary, his mother. "
"And they fell down and worshiped him. "
"Opening their treasures, they offered to him gifts: gold, frankincense, and myrrh. "
"Being warned in a dream not to return to Herod. "
"They went back to their own country another way. "
"Now when they had departed, behold, an angel of the Lord appeared to Joseph in a dream. "
"Saying, Arise and take the young child and his mother. "
"And flee into Egypt and stay there until I tell you. "
"For Herod will seek the young child to destroy him. "
"He arose and took the young child and his mother by night. "
"And departed into Egypt. "
"And was there until the death of Herod. "
"That it might be fulfilled which was spoken by the Lord through the prophet. "
"Out of Egypt I called My Son. "
"Then Herod, when he saw that he was mocked by the wise men, was exceedingly angry. "
"And sent out and killed all the male children who were in Bethlehem. "
"And in all the surrounding countryside, from two years old and under. "
"According to the exact time which he had learned from the wise men. "
"Then that which was spoken by Jeremiah the prophet was fulfilled. "
"Saying, A voice was heard in Rama, lamentation, weeping, and great mourning. "
"Rachel weeping for her children, and she would not be comforted. "
"Because they are no more. "
"But when Herod was dead, behold, an angel of the Lord appeared in a dream to Joseph in Egypt. "
"Saying, Arise and take the young child and his mother. "
"And go into the land of Israel. "
"For those who sought the young child's life are dead. "
"He arose and took the young child and his mother. "
"And came into the land of Israel. "
"But when he heard that Archelaus was reigning over Judea in the place of his father Herod. "
"He was afraid to go there. "
"Being warned in a dream, he withdrew into the region of Galilee. "
"And came and lived in a city called Nazareth. "
"That it might be fulfilled which was spoken through the prophets. "
"- He will be called a Nazarene! "
)

# ── 3. NARRATOR VOICE ──────────────────────────────────────────────────────
# ⚠️  Must match the LANGUAGE of TEXTO_ORACAO above (Edge TTS reads the text
#    with that voice's pronunciation — wrong voice = strange accent).
#
#    🇧🇷 Portuguese:  "pt-BR-AntonioNeural" (m) · "pt-BR-FranciscaNeural" (f)
#    🇺🇸 English:      "en-US-GuyNeural" (m) · "en-US-JennyNeural" (f)
#                      "en-US-ChristopherNeural" (m, deep/narrator)
#    🇪🇸 Spanish:      "es-ES-AlvaroNeural" (m) · "es-ES-ElviraNeural" (f)
#    🇫🇷 French:       "fr-FR-HenriNeural" (m) · "fr-FR-DeniseNeural" (f)
#    🇨🇳 Chinese:      "zh-CN-YunxiNeural" (m) · "zh-CN-XiaoxiaoNeural" (f)
#
VOZ_EDGE = "en-US-GuyNeural"

# ── 3b. MASTER LANGUAGE ────────────────────────────────────────────────────
# Language of TEXTO_ORACAO/VOZ_EDGE above -- must match. Used to find the
# right Whisper SRT later (caption-single-generate.ipynb).
IDIOMA_MESTRE = "en"

# ── 6. IMAGE SHEET (Pixabay photos) ────────────────────────────────────────
# Google Sheet ID from its URL
# (https://docs.google.com/spreadsheets/d/THIS_PART_HERE/edit...). Must have
# "Imagem" (direct image URL), "Autor", and a status column matching
# NOME_COLUNA_STATUS_PLANILHA below.
ID_PLANILHA_IMAGENS_DRIVE = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"  # pixabay-image-stock
NOME_COLUNA_STATUS_PLANILHA = "Downloading Ok"

# Each image is shown still for this many seconds before switching to the
# next one (sequential -- next UNUSED row in the sheet, same anti-repeat
# logic as the video mode).
DURACAO_CLIPE = 5

# ── 7. DRIVE ROOT FOLDER ──────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project


# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION (image background)")
print("=" * 60)
print(f"   Name:            {NOME_ORACAO}")
print(f"   Voice:           {VOZ_EDGE}")
print(f"   Master language:  {IDIOMA_MESTRE}")
print(f"   Narration volume:  {VOLUME_NARRACAO}")
print(f"   Music volume:      {VOLUME_MUSICA}")
print(f"   Speed:           {VELOCIDADE_AUDIO}x")
print(f"   Image sheet:     {ID_PLANILHA_IMAGENS_DRIVE or '⚠️ NOT SET'}")
print(f"   Seconds/image:   {DURACAO_CLIPE}")
print(f"   Drive root:      {PASTA_DRIVE_RAIZ}")
print(f"   Text:            {TEXTO_ORACAO[:60]}...")
print("=" * 60)
print("✅ Configuration ready — proceed to Script/Audio (optional) and Initialization")


⚙️  CONFIGURATION
   Name:            40_Matt_02
   Voice:           en-US-GuyNeural
   Narration volume:  1.0
   Music volume:      0.25
   Speed:           1.0x
   Background:      imagem (no sheet ID set!)
   Drive root:      narrated_video
   Text:            Now when Jesus was born in Bethlehem of Judea, in the days o...
✅ Configuration ready — proceed to Script/Audio (optional) and Initialization


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📄 SCRIPT AND AUDIO FROM DRIVE (optional)                       ║
# ║  If present on Drive, these REPLACE TEXTO_ORACAO and skip narration ║
# ║  Script: [NAME]_roteiro.txt                                       ║
# ║  Audio:  [NAME]_audio.wav  (or .mp3/.m4a/.ogg — auto-converted)   ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil, subprocess

PASTA_VIDEO_DRIVE = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/videos/{NOME_ORACAO}")
PASTA_VIDEO_DRIVE.mkdir(parents=True, exist_ok=True)

# ── Script (text) ─────────────────────────────────────────────────────────
roteiro_drive = PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_roteiro.txt"
if roteiro_drive.exists():
    TEXTO_ORACAO = roteiro_drive.read_text(encoding="utf-8").strip()
    print(f"✅ Script loaded from Drive: {roteiro_drive.name} ({len(TEXTO_ORACAO)} characters)")
else:
    print(f"ℹ️  No script at {roteiro_drive.name} — using the text from the Configuration cell")

# ── Audio — accepts .wav, .mp3, .m4a, .ogg; converts to wav if needed ──────
audio_local = Path(f"/content/{NOME_ORACAO}_audio.wav")
EXTENSOES_AUDIO = [".wav", ".mp3", ".m4a", ".ogg", ".flac"]

audio_drive = next(
    (PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_audio{ext}" for ext in EXTENSOES_AUDIO
     if (PASTA_VIDEO_DRIVE / f"{NOME_ORACAO}_audio{ext}").exists()),
    None,
)

if audio_drive is None:
    print(f"ℹ️  No audio at {PASTA_VIDEO_DRIVE.name}/{NOME_ORACAO}_audio.* — will be generated by Edge TTS")
elif audio_drive.suffix == ".wav":
    shutil.copy2(audio_drive, audio_local)
    print(f"✅ Audio (.wav) loaded from Drive: {audio_drive.name} ({audio_local.stat().st_size/1_048_576:.2f} MB)")
else:
    print(f"🔄 Audio found as {audio_drive.suffix}: {audio_drive.name} — converting to .wav...")
    resultado = subprocess.run(
        ["ffmpeg", "-y", "-i", str(audio_drive), "-ar", "44100", "-ac", "2", str(audio_local)],
        capture_output=True, text=True,
    )
    if audio_local.exists():
        print(f"✅ Converted: {audio_local.name} ({audio_local.stat().st_size/1_048_576:.2f} MB)")
        shutil.copy2(audio_local, PASTA_VIDEO_DRIVE / audio_local.name)
    else:
        print(f"❌ Conversion failed — check the original file")
        print(resultado.stderr[-800:])


✅ Script loaded from Drive: 40_Matt_02_roteiro.txt (3259 characters)
✅ Audio (.wav) loaded from Drive: 40_Matt_02_audio.wav (33.41 MB)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️ DOWNLOAD AUTO-DUBBED AUDIO (YouTube, optional)               ║
# ║                                                                    ║
# ║  Pulls an auto-dubbed audio track from a YouTube video (via       ║
# ║  yt-dlp) instead of generating narration with Edge TTS.           ║
# ║                                                                    ║
# ║  Saves as [NAME]_audio.wav — the "Narration" cell below detects   ║
# ║  it already exists and skips TTS generation automatically.        ║
# ║                                                                    ║
# ║  TOTALLY OPTIONAL — skip this cell to use Edge TTS normally.      ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from pathlib import Path

URL_DUBLAGEM    = "https://www.youtube.com/watch?v=4vTN7tBG3a8"  # video URL with auto-dubbing
IDIOMA_DUBLAGEM = "en"    # language code of the audio track you want (e.g. "pt", "en", "es", "fr")
FORMAT_ID_MANUAL = ""     # leave blank to try automatic; if it fails, see the printed
                          # list below and paste the exact track ID here (e.g. "233-1")

audio_local = Path(f"/content/{NOME_ORACAO}_audio.wav")

print("📋 Available audio tracks in this video:")
lista = subprocess.run(["yt-dlp", "-F", URL_DUBLAGEM], capture_output=True, text=True)
for linha in lista.stdout.splitlines():
    if "audio only" in linha:
        print("  ", linha)
print()

formato = FORMAT_ID_MANUAL.strip() or f"ba[language^={IDIOMA_DUBLAGEM}]/bestaudio[language^={IDIOMA_DUBLAGEM}]"
print(f"🎙️  Downloading audio track (format: {formato})...")

resultado = subprocess.run(
    ["yt-dlp", "-f", formato, "--extract-audio", "--audio-format", "wav",
     "-o", "temp_dublagem.%(ext)s", URL_DUBLAGEM],
    capture_output=True, text=True,
)

temp_wav = Path("temp_dublagem.wav")
if temp_wav.exists():
    temp_wav.replace(audio_local)
    print(f"✅ Dubbed audio saved: {audio_local.name} ({audio_local.stat().st_size/1_048_576:.2f} MB)")
else:
    print("❌ Couldn't find a matching track automatically.")
    print("   Check the track list printed above, copy the ID of the track you want")
    print("   (left column, e.g. '233-1') and paste it in FORMAT_ID_MANUAL. Error detail:")
    print(resultado.stderr[-800:])


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint

config = PipelineConfig(
    NOME_ORACAO                  = NOME_ORACAO,
    PASTA_DRIVE_RAIZ              = PASTA_DRIVE_RAIZ,
    TEXTO_ORACAO                  = TEXTO_ORACAO,
    VOZ_EDGE                      = VOZ_EDGE,
    IDIOMA_MESTRE                  = IDIOMA_MESTRE,
    VOLUME_NARRACAO               = VOLUME_NARRACAO,
    VOLUME_MUSICA                 = VOLUME_MUSICA,
    VELOCIDADE_AUDIO              = VELOCIDADE_AUDIO,
    MODO_CLIPE                    = "imagem",
    ID_PLANILHA_IMAGENS_DRIVE     = ID_PLANILHA_IMAGENS_DRIVE,
    NOME_COLUNA_STATUS_PLANILHA   = NOME_COLUNA_STATUS_PLANILHA,
    DURACAO_CLIPE                  = DURACAO_CLIPE,
)

pipeline = VideoPipeline(config)
cp       = Checkpoint(nome_oracao=config.NOME_ORACAO)  # isolated per video — see checkpoint.py

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:       {config.NOME_ORACAO}")
print(f"   Folder:      {config.pasta_oracao}")
print(f"   Checkpoint:  {cp.proxima_fase_pendente() or 'all done'}")
print("=" * 60)
print(config.resumo())


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎤 NARRATION — generate audio with Edge TTS                     ║
# ║  Skips generation if the audio already exists (Drive/manual/etc.) ║
# ╚══════════════════════════════════════════════════════════════════╝

audio = pipeline.gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🖼️🏷️ CONVERT IMAGES + CREDIT + LOGO — use what's ready on Drive, ║
# ║       download only what's missing                                ║
# ║                                                                    ║
# ║  Converts photos from the image sheet (Configuration cell) into   ║
# ║  still video segments, DURACAO_CLIPE seconds each, pulling the    ║
# ║  next UNUSED row in sequence (anti-repeat, same logic as video     ║
# ║  No verse matching in this notebook -- see                       ║
# ║  video-base-imagem-versiculo.ipynb for that.                      ║
# ║                                                                    ║
# ║  Checkpoint is isolated per video (checkpoint_[NAME].json).       ║
# ╚══════════════════════════════════════════════════════════════════╝

clipes = pipeline.baixar_clipes_imagem()

print(f"\n✅ {len(clipes)} image segment(s) ready in clipes_cortados/ ({DURACAO_CLIPE}s each)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  💾 SAVE CLIPS TO DRIVE (optional)                               ║
# ║                                                                    ║
# ║  Copies this video's cut/credited clips to the shared pool         ║
# ║  (assets/clipes/), so future videos can reuse them instead of      ║
# ║  downloading from the sheet again.                                 ║
# ║                                                                    ║
# ║  TOTALLY OPTIONAL — safe to skip.                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

n_salvos = pipeline.salvar_clipes_no_drive(clipes)
print(f"💾 {n_salvos} new clip(s) saved to {config.pasta_assets_clipes}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎬 VIDEO BASE — concatenate + narration + background music      ║
# ║  Clips arrive here already cut and credited (previous cell);      ║
# ║  this step just joins everything and mixes in the audio.          ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from ffmpeg_utils import obter_duracao

video_base = pipeline.criar_video_base(clipes)

if video_base and video_base.exists():
    duracao_final  = obter_duracao(video_base)
    duracao_audio  = obter_duracao(Path(config.NOME_AUDIO))
    print(f"✅ VIDEO BASE: {video_base.name} ({video_base.stat().st_size/1_048_576:.1f} MB, {duracao_final:.1f}s)")
    if abs(duracao_final - duracao_audio) > 1.0:
        print(f"⚠️  Warning: video ({duracao_final:.1f}s) doesn't match the audio ({duracao_audio:.1f}s) — please check.")
    else:
        print(f"✅ Duration matches the audio ({duracao_audio:.1f}s)")
else:
    print("⚠️  Video base was not generated — check the logs above.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW VIDEO BASE                                           ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from IPython.display import Video, display

video_base = Path(config.NOME_VIDEO_BASE)
if video_base.exists():
    print(f"🎬 {video_base.name}  ({video_base.stat().st_size/1_048_576:.1f} MB)")
    display(Video(str(video_base), embed=True, width=800))
else:
    print(f"❌ Video base not found: {video_base}")
    print("   Run the clip-cutting and video-base cells first.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — VIDEO BASE                                        ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
from google.colab import files

video_base = Path(config.NOME_VIDEO_BASE)
if video_base.exists():
    print(f"📥 Downloading {video_base.name} ({video_base.stat().st_size/1_048_576:.1f} MB)...")
    files.download(str(video_base))
else:
    print(f"❌ Video base not found: {video_base}")


### 🔧 Utilities

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 SELECTIVE CLEANUP                                            ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil

def limpeza_seletiva():
    print("=" * 60)
    print("🧹 SELECTIVE CLEANUP")
    print("=" * 60)
    print("  1 - 🎵 Audio (.wav, .mp3)")
    print("  2 - 🎬 Generated videos (base, clips)")
    print("  3 - 📌 Checkpoint")
    print("  4 - 📁 Temp folders (clipes_cortados/, temp_raw/)")
    print("  5 - 🔥 EVERYTHING (1-4)")
    print("  0 - Cancel")
    escolha = input("\nEnter numbers separated by commas: ").strip()
    if escolha == '0':
        return
    opcoes = [int(x.strip()) for x in escolha.split(',')]
    cont = 0

    if 1 in opcoes or 5 in opcoes:
        for f in Path('.').glob('*_audio.wav'):
            f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

    if 2 in opcoes or 5 in opcoes:
        for pattern in ['*_video_base.mp4', 'video_com_audio.mp4', 'video_sem_audio.mp4']:
            for f in Path('.').glob(pattern):
                f.unlink(); cont += 1; print(f"   🗑️ {f.name}")

    if 3 in opcoes or 5 in opcoes:
        for cp_file in Path('.').glob('checkpoint*.json'):
            cp_file.unlink(); cont += 1; print(f"   🗑️ {cp_file.name}")

    if 4 in opcoes or 5 in opcoes:
        for pasta in ['clipes_cortados', 'temp_raw', '__pycache__']:
            p = Path(pasta)
            if p.exists():
                shutil.rmtree(p); cont += 1; print(f"   🗑️ {pasta}/")

    print(f"\n✅ {cont} item(s) removed")

limpeza_seletiva()
